# Finetuning Granite Speech (LoRA)


In [1]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


## Setup

In [1]:
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

model.config.use_cache = False


Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [2]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def read_manifest(path):
    """Read a JSONL manifest into a list of dicts."""
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)
dev_data = read_manifest(MANIFEST_DEV)

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

1000 train samples, 100 dev samples
{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131007-0900-PLENARY-19-en_20131007-21:26:04_7.wav', 'text': 'this could be done by looking more specifically into the suggested fallback clause in article twenty one in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions', 'prompt': 'Vocabulary: article twenty one.'}


## Prompt Construction

In [3]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    """Apply the Granite Speech chat template to a transcription request."""
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "can you transcribe the speech into a written format?"

def build_user_prompt(sample):
    """Default transcription prompt, with keywords appended when the manifest has them."""
    prompt = sample.get("prompt")
    if prompt:
        return f"{DEFAULT_PROMPT} Keywords: {prompt}"
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Collate Function (Audio + Prompt + Target Text)

In [5]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]

    # Processor pads input_ids, attention_mask, input_features and input_features_mask
    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)   # CPU tensors

    # Append the target transcript as supervision and mask the prompt part
    target_ids = [processor.tokenizer(sample["text"], add_special_tokens=False,
                                      return_tensors="pt").input_ids[0] for sample in batch]
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t   # ignore prompt/audio tokens (-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

this could be done by looking more specifically into the suggested fallback clause in article twenty one in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions


## Training Configuration

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [7]:
import jiwer
from tqdm import tqdm

def compute_dev_wer():
    """Greedy-transcribe the dev set with the current model and compute WER"""
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)



In [8]:
print("Dev WER before training:", compute_dev_wer())

100%|██████████| 100/100 [01:24<00:00,  1.19it/s]

Dev WER before training: 0.052007927069935525


## Train

In [9]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    """Print loss and learning rate at every logging step"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,4.114339,3.273510
2,4.189428,3.846425


step 1: {'loss': 6.02384614944458, 'grad_norm': 2176.0, 'learning_rate': 2e-06, 'epoch': 0.004}
step 10: {'loss': 5.785183376736111, 'grad_norm': 808.0, 'learning_rate': 1.9639999999999997e-06, 'epoch': 0.04}
step 20: {'loss': 3.71424560546875, 'grad_norm': 1152.0, 'learning_rate': 1.9239999999999997e-06, 'epoch': 0.08}
step 30: {'loss': 3.62010498046875, 'grad_norm': 352.0, 'learning_rate': 1.8839999999999999e-06, 'epoch': 0.12}
step 40: {'loss': 9.668412017822266, 'grad_norm': 752.0, 'learning_rate': 1.844e-06, 'epoch': 0.16}
step 50: {'loss': 5.6417289733886715, 'grad_norm': 284.0, 'learning_rate': 1.804e-06, 'epoch': 0.2}
step 60: {'loss': 5.164793395996094, 'grad_norm': 476.0, 'learning_rate': 1.764e-06, 'epoch': 0.24}
step 70: {'loss': 6.019762420654297, 'grad_norm': 2256.0, 'learning_rate': 1.724e-06, 'epoch': 0.28}
step 80: {'loss': 4.1205894470214846, 'grad_norm': 1488.0, 'learning_rate': 1.6839999999999999e-06, 'epoch': 0.32}
step 90: {'loss': 4.836156845092773, 'grad_norm': 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

step 260: {'loss': 2.9982015609741213, 'grad_norm': 436.0, 'learning_rate': 9.64e-07, 'epoch': 1.04}
step 270: {'loss': 3.2890323638916015, 'grad_norm': 278.0, 'learning_rate': 9.24e-07, 'epoch': 1.08}
step 280: {'loss': 3.998279571533203, 'grad_norm': 616.0, 'learning_rate': 8.839999999999999e-07, 'epoch': 1.12}
step 290: {'loss': 2.866984176635742, 'grad_norm': 286.0, 'learning_rate': 8.439999999999999e-07, 'epoch': 1.16}
step 300: {'loss': 2.7098983764648437, 'grad_norm': 342.0, 'learning_rate': 8.04e-07, 'epoch': 1.2}
step 310: {'loss': 3.64443359375, 'grad_norm': 350.0, 'learning_rate': 7.64e-07, 'epoch': 1.24}
step 320: {'loss': 3.6275470733642576, 'grad_norm': 1504.0, 'learning_rate': 7.24e-07, 'epoch': 1.28}
step 330: {'loss': 3.404189682006836, 'grad_norm': 506.0, 'learning_rate': 6.84e-07, 'epoch': 1.32}
step 340: {'loss': 4.905161666870117, 'grad_norm': 312.0, 'learning_rate': 6.44e-07, 'epoch': 1.3599999999999999}
step 350: {'loss': 3.9820606231689455, 'grad_norm': 352.0, '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

step 500: {'train_runtime': 583.5215, 'train_samples_per_second': 3.427, 'train_steps_per_second': 0.857, 'total_flos': 4702778760975696.0, 'train_loss': 4.023499001502991, 'epoch': 2.0}


TrainOutput(global_step=500, training_loss=4.023499001502991, metrics={'train_runtime': 583.5215, 'train_samples_per_second': 3.427, 'train_steps_per_second': 0.857, 'total_flos': 4702778760975696.0, 'train_loss': 4.023499001502991, 'epoch': 2.0})

In [ ]:
print("Dev WER AFTER training:", compute_dev_wer())

 29%|██▉       | 29/100 [00:19<00:53,  1.32it/s]

## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [ ]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")